# SMR multilink modelization


Combined Heat and Power (CHP) with fixed heat-power ratio. 

the following code is based in a PyPSA CHP multilink [example](https://pypsa.readthedocs.io/en/latest/examples/chp-fixed-heat-power-ratio.html) that demonstrates a [Link](https://pypsa.readthedocs.io/en/latest/user-guide/components.html#link) component with more than one [bus](https://pypsa.readthedocs.io/en/latest/user-guide/components.html#bus) output (“bus2” in this case).

The SMR_CHP must be heat-following because there is no other supply of heat to the bus "Industry Electricity heat”.

Hypothesis for this example: 
- Simulation for 10 snapshots, that represent 10 days.
- Suppose 8000 MWh of demand for chemical industry in spain, during 10 days period. (800 MWh energy demand per day, and 800/24h=33MW of power demand)
- Grid is allowed to offer 30% of demand for each day: 800*0.3= 240MWh

In [10]:
import numpy as np
# import numpy_financial as npf
import matplotlib.pyplot as plt
import pandas as pd
import pypsa

- **Efficiency**: Efficiency of power transfer from bus0 to bus1. (Can be time-dependent to represent temperature-dependent Coefficient of Performance of a heat pump from an electric to a heat bus.)

In [ ]:
network.links

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,up_time_before,down_time_before,ramp_limit_up,ramp_limit_down,ramp_limit_start_up,ramp_limit_shut_down,p_nom_opt,bus2,efficiency2,efficiency1
Link,,,,,,,,,,,,,,,,,,,,,
Gas_CHP,Spain gas,Industry Electricity,,gas,1.00,True,0,inf,0.0,0.0,...,1,0,NaN,NaN,1.0,1.0,0.0,Industry Heat,0.45,0.35
SMR_CHP,Nuclear_Fuel,Industry Electricity,,nuclear,0.35,True,0,inf,0.0,0.0,...,1,0,NaN,NaN,1.0,1.0,0.0,Industry Heat,0.45,NaN


In [ ]:
'''#PRELIMINARIES
# marginal costs in EUR/MWh
marginal_costs = {"Wind": 0, "Hydro": 0, "Coal": 30, "Gas": 60, "Oil": 80}

# power plant capacities (nominal powers in MW) in each country (not necessarily realistic)
power_plant_p_nom = {
    "South Africa": {"Coal": 35000, "Wind": 3000, "Gas": 8000, "Oil": 2000},
    "Mozambique": {"Hydro": 1200,},
    "Swaziland": {"Hydro": 600,},
}
# transmission capacities in MW (not necessarily realistic)
transmission = {
    "South Africa": {"Mozambique": 500, "Swaziland": 250},
    "Mozambique": {"Swaziland": 100},
}
# country electrical loads in MW (not necessarily realistic)
loads = {"South Africa": 42000, "Mozambique": 650, "Swaziland": 250}
country = "South Africa"

network = pypsa.Network()

network.add("Bus", country)

for tech in power_plant_p_nom[country]:
    network.add(
        "Generator",
        f"{country} {tech}",
        bus=country,
        p_nom=power_plant_p_nom[country][tech],
        marginal_cost=marginal_costs[tech],
    )


network.add("Load", f"{country} load", bus=country, p_set=loads[country])
# Run optimisation to determine market dispatch
network.optimize()
network.generators
# print the load active power (P) consumption
network.loads_t.p
# # print the generator active power (P) dispatch
network.generators_t.p
# print the clearing price (corresponding to gas)
network.buses_t.marginal_price

#EXTRA Parte2: 
# # ADD HEAT DUMP FOR EXCESS HEAT
network.add("Load", "Heat Dump",
       bus="Industry Heat",
       p_set=-1000,  # Allow excess heat absorption
       p_max_pu=0)   # No actual heat demand here

limit = -50 # can go to -50 (co2_emissions variable limit)
network.add("GlobalConstraint", # Add a global CO2 constraint
      "co2_limit",              #arbitrary name
      sense="<=",               #attribute of the constraint
      carrier_attribute="co2_emissions",  # what variable is affected
      constant=limit)                #Our constant limit created before'''

In [ ]:
network = pypsa.Network() # Creates PyPSA network object called 'network'
# snapshots=list(np.arange(1,8761,1, dtype=int))
network.set_snapshots(range(10)) #change range(10) by 'snapshots' to have 8760 hours to optimze and use historic hourly input DataFiles. 


# 1. DEFINE CARRIERS FIRST
carriers = ["electricity", "heat", "nuclear", "gas"]
for c in carriers:
    network.add("Carrier", c)

# Industry Electricity system
network.add("Bus", "Industry Electricity", carrier="electricity")
network.add("Load", "Industry Electricity Load", 
       bus="Industry Electricity", 
       p_set=33)  # 33 MW electricity demand

# Industry Heat system
network.add("Bus", "Industry Heat", carrier="heat")
network.add("Load", "Industry Heat Load", 
       bus="Industry Heat", 
       p_set=3)  # 3 MW demand

# Energy Fuel buses
network.add("Bus", "Nuclear_Fuel", carrier="nuclear")
network.add("Bus", "Spain_gas", carrier="gas")
network.add("Bus", "Spain_grid", carrier="electricity")

# Store of Uranium with fuel marginal cost
network.add("Store", "Nuclear_Fuel",
       bus="Nuclear_Fuel",
       carrier="nuclear",
       e_initial=50,  # MWh-th
       e_cyclic=True, #if True, then e_initial is ignored and the initial energy is set to the final energy for the group of snapshots in the OPF.
       e_nom=50,
       marginal_cost=10) #$ /MWh *térmico* (uranio)

# Store of Gas  with fuel marginal cost
network.add("Store", "Spain_gas",
       bus="Spain_gas",
       carrier="gas",
       e_initial=50, # MWh-th
       e_cyclic=True, #TRUE: e_initial is set to the final energy for the group of snapshots in the OPF.
       e_nom=50,
       marginal_cost=30)  # $/MWh *térmico* (gas natural))

# Store of electricity with fuel marginal cost
network.add("Store", "Spain_grid",
       bus="Spain_grid",
       carrier="electricity",
       e_nom_extendable=True, # allow capacity e_nom to be extended in OPF.
       e_nom_min= 50,         # MWh 
       e_nom_max= 240,    # MWh (Example Hypothesis)
       marginal_cost=50, # $/MWh Grid cost of electricity
       capital_cost=1)   #Fixed period costs of extending e_nom by 1 MWh, including periodized investment costs and periodic fixed O&M costs (e.g. annuitized investment costs).

# Enlace Gas_CHP (sin costo marginal adicional)
network.add("Link", "Gas_CHP",
    bus0="Spain_gas",
    bus1="Industry Electricity",
    bus2="Industry Heat",
    carrier="gas",  #Energy carrier transported by the link
    p_nom_extendable=True,
    capital_cost=1e6,  # 1 millón $/MW
    marginal_cost=0,  # $/MWh no aditional cost for fuel (computed in store)
    efficiency=0.30, # 30% bus0(Gas) → bus1(electricity))
    efficiency2=0.35       # Heat
)
# Enlace SMR_CHP (sin costo marginal adicional)
network.add("Link", "SMR_CHP",
    bus0="Nuclear_Fuel",
    bus1="Industry Electricity",
    bus2="Industry Heat",
    carrier="nuclear",
    p_nom_extendable=True,
    capital_cost=12e6,    # 12 millones $/MW
    marginal_cost=0,  # $/MWh no aditional cost for fuel (computed in nuclear fuel store)
    efficiency=0.30,      # 35% uranio → electricidad
    efficiency2=0.45      # Heat
)
# Enlace Grid (sin costo marginal adicional)
network.add("Link", "Grid_CHP",
    bus0="Spain_grid",
    bus1="Industry Electricity",
    carrier="electricity",
    p_nom_extendable=True,
    capital_cost=12e6,    # 12 millones $/MW
    marginal_cost=0,  # $/MWh no aditional cost for fuel (computed in nuclear fuel store)
    efficiency=0.95)      #  electricity transport

Index(['Grid_CHP'], dtype='object')

In [30]:
network.optimize()

INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.1s
Status: warning
Termination condition: infeasible
Solution: 0 primals, 0 duals
Objective: nan
Solver model: available
Solver message: infeasible



('warning', 'infeasible')

In [ ]:
# print(network.stores_t.e)

AttributeError: p

In [26]:
#network #view the network network

In [27]:
 network.buses

,v_nom,type,x,y,carrier,unit,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network
Bus,,,,,,,,,,,,
Industry Electricity,1.0,,0.0,0.0,electricity,,1.0,0.0,inf,PQ,,
Industry Heat,1.0,,0.0,0.0,heat,,1.0,0.0,inf,PQ,,
Nuclear_Fuel,1.0,,0.0,0.0,nuclear,,1.0,0.0,inf,PQ,,
Spain gas,1.0,,0.0,0.0,gas,,1.0,0.0,inf,PQ,,


There are no generators or storage_units because they have been modelled as links and buses.

**Note** that negative values for network.links_t.p1 indicate that the link is injecting power into bus1 (This aligns with PyPSA's [convention](https://pypsa.readthedocs.io/en/latest/user-guide/design.html#sign-conventions) where power injected into a bus is considered negative, while power withdrawn from a bus is positive). 

- network.links_t.p0 # energy flows through the input bus (bus0) in each link (for each snapshot) CONSUMO DE COMBUSTIBLE
- network.links_t.p1 # energy flows through the output bus1 (bus1) in each link (for each snapshot) ENERGY GENERATION (CONVERSION)
- network.links_t.p2 # energy flows through the output bus2 (bus2) in each link (for each snapshot) ENERGY GENERATION (CONVERSION)

In [28]:
# network.generators.index 
#network.storage_units.index
# network.loads_t.p
print("SMR Electricity Output:", network.links_t.p1["SMR_CHP"].sum())  #  MW
print("SMR Heat Output:", network.links_t.p2["SMR_CHP"].sum())         #  MW
print("SMR Uranium Fuel Consumption:", network.links_t.p0["SMR_CHP"].sum())   

KeyError: 'SMR_CHP'

In [29]:
'''print("Gas_CHP Electricity Output:", network.links_t.p1["Gas_CHP"].sum())
print("Gas_CHP Heat Output:", network.links_t.p2["Gas_CHP"].sum())         #  MW
print("Gas_CHP Gas Consumption:", network.links_t.p0["Gas_CHP"].sum())'''

'print("Gas_CHP Electricity Output:", network.links_t.p1["Gas_CHP"].sum())\nprint("Gas_CHP Heat Output:", network.links_t.p2["Gas_CHP"].sum())         #  MW\nprint("Gas_CHP Gas Consumption:", network.links_t.p0["Gas_CHP"].sum())'

# End of the fuctional example

<!-- 
Starts SMR model versión2 with network `n2`

## Note: 
About de network initialization (can be useful for some future research)

**Network Initialization**: `n1 = pypsa.Network(url)`
   - Creates a PyPSA `Network` object `n1`
   - Automatically downloads and loads the pre-configured energy system model containing:
     - Buses (energy nodes)
     - Generators (power plants)
     - Loads (energy demands)
     - Transmission lines
     - Time-dependent constraints

Key Features of This Approach:
- **Reproducibility**: Loads a standardized benchmark model
- **Time-Saving**: Avoids manual network construction (~1000+ lines of setup code)
- **Research-Ready**: Contains realistic European power system data (likely from [PyPSA-Eur](https://pypsa-eur.readthedocs.io/)) -->